<a href="https://colab.research.google.com/github/goyoh/arena/blob/master/%E4%BC%81%E6%A5%AD%E3%83%AA%E3%82%B9%E3%83%88%E3%81%8B%E3%82%89ESG%E8%A6%81%E4%BB%B6%E3%83%BB%E8%A6%81%E6%9C%9B%E3%82%92%E5%8F%8E%E9%9B%86.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# -*- coding: utf-8 -*-
"""
【Gemini API版】企業のESG要件・要望収集エージェント

このプログラムは、Googleスプレッドシートから企業リストを読み込み、
Gemini APIを使って各企業のESGに関する「要件」「要望」「目標」をWeb上から収集し、
結果を別のシートに書き出すためのツールです。

使い方:
1. Colabの「シークレット」に `GEMINI_API_KEY` という名前でご自身のAPIキーを保存します。
2. 「Step 4」のフォームで使用するモデル、スプレッドシートのURL、
   シート名、および読み書きする列を指定します。
3. まず、Step 1のセルのみを実行します。
4. Step 1の出力メッセージに従い、Colabのメニューから「ランタイム」>「ランタイムを再起動」
   を選択し、手動でランタイムを再起動します。
5. ランタイムが再接続された後、Step 2以降のセルを順番に実行することで、
   全ての処理が開始されます。（Step 1は再実行しないでください）
"""

# ==============================================================================
# Step 1: 必要なライブラリのインストール
# ==============================================================================
!pip install --upgrade gspread "google-genai>=1.33.0" gspread-dataframe google-auth-oauthlib "pandas==2.2.2" thefuzz python-Levenshtein -q

print("\nライブラリのインストール・更新が完了しました。")

# ==============================================================================
# Step 2: ライブラリのインポートとGoogleサービスへの認証
# ==============================================================================
import pandas as pd
from google import genai
from google.genai import types
import gspread
from gspread_dataframe import set_with_dataframe
from google.colab import auth, userdata
from google.auth import default
import json
import time
import datetime
import requests
import html
import re
import unicodedata
from urllib.parse import urlparse, urlunparse, parse_qsl, urlencode
from thefuzz import fuzz

# バージョン情報を出力して、ライブラリが正しく更新されているか確認
try:
    print(f"google-genai version: {genai.__version__}")
except Exception as e:
    print(f"ライブラリのバージョン確認中にエラー: {e}")


# Google Colab上でGoogleサービス（Drive, Spreadsheetなど）への認証を行う
try:
    auth.authenticate_user()

    # 認証情報を使ってgspreadのクライアントを初期化
    creds, _ = default()
    gc = gspread.authorize(creds)

    print("ライブラリのインポートとGoogleサービスへの認証が完了しました。")

except Exception as e:
    print(f"認証中にエラーが発生しました。ランタイム再起動後の2回目の実行であることを確認してください。エラー: {e}")

# Gemini APIのHTTPオプション設定（タイムアウト延長など）
http_options = types.HttpOptions(
    client_args={'timeout': 180}  # 秒。httpxのtimeoutにそのまま渡されます
)

# ==============================================================================
# Step 3: APIキーの読み込みとGeminiクライアントの設定
# ==============================================================================
# @title Gemini API セットアップ
client = None # Geminiクライアントを格納する変数を初期化
# ColabのシークレットからAPIキーを読み込み
try:
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
    client = genai.Client(api_key=GEMINI_API_KEY, http_options=http_options)
    print("Gemini APIキーの読み込みと設定が完了しました。")
except userdata.SecretNotFoundError:
    print("エラー: Colabのシークレットに 'GEMINI_API_KEY' が見つかりません。")
except Exception as e:
    print(f"APIキーの設定中にエラーが発生しました: {e}")

# 使用するモデルを選択
# @markdown 使用する生成モデルを選択してください。
MODEL_NAME = "gemini-2.5-flash"  # @param ["gemini-2.5-pro", "gemini-2.5-flash"]


# ==============================================================================
# Step 4: スプレッドシート情報の設定
# ==============================================================================
# @title スプレッドシートと処理範囲の設定
# @markdown **読み込み元と書き込み先のシート情報を指定してください。**
SPREADSHEET_URL = "https://docs.google.com/spreadsheets/d/1JjqVmk4H_Dp0uSdqYEY_6uWNkjX2wUUuf1gqwc4_6e0/edit?usp=sharing"  # @param {type:"string"}
# @markdown **読み込み元シート名**
READ_SHEET_NAME = "Companies"  # @param {type:"string"}
# @markdown **書き込み先シート名**
WRITE_SHEET_NAME = "Requirements"  # @param {type:"string"}
# @markdown **教師データシート名**
LABELED_SHEET_NAME = "Labeled" #@param {type:"string"}
# @markdown **少数ショット学習を有効にする**
ENABLE_FEW_SHOT_LEARNING = True #@param {type:"boolean"}
# @markdown **少数ショット学習で使用する教師データの最大行数（0以下で無制限）**
FEW_SHOT_LIMIT = -1 #@param {type:"number"}
# @markdown **コンテキストキャッシュの有効時間（時間単位、0で無効）**
CACHE_TTL_HOURS = 1 #@param {type:"number"}
# @markdown **重複判定の類似度しきい値（%単位、0〜100）**
SIMILARITY_THRESHOLD = 92 #@param {type:"number"}
# @markdown ---
# @markdown **企業名(CompanyName)が記載されている列 (読み込み元)**
COMPANY_NAME_COLUMN = "B"  # @param {type:"string"}
# @markdown **国名(Country)が記載されている列 (読み込み元)**
COUNTRY_COLUMN = "C"  # @param {type:"string"}
# @markdown **住所(Address)が記載されている列 (読み込み元)**
ADDRESS_COLUMN = "D"  # @param {type:"string"}

VERTEX_HOSTS = (
    "https://vertexaisearch.cloud.google.com/",
    "https://vertexaisearch.google.com/",
)
# 明示検索の設定
ENABLE_PRE_DISCOVERY = True   # 先行して複数クエリでURLを拾う
DISCOVERY_RESULT_LIMIT = 3    # 各クエリで拾う最大URL数
PRE_DISCOVERY_MAX_QUERIES = 30  # 任意

# 動的なキーワード戦略のための対象都市リスト
TARGET_CITIES = ["Tokyo", "New York", "London", "Singapore", "Hong Kong", "Sydney"]


# ==============================================================================
# Step 5: スプレッドシートからデータの読み込み
# ==============================================================================
df = None
company_col_name, address_col_name, country_col_name, company_id_col_name = None, None, None, None
primary_categories, tertiary_categories, labeled_df = [], [], None
few_shot_json_examples = ""


try:
    if SPREADSHEET_URL and 'gc' in locals():
        spreadsheet = gc.open_by_url(SPREADSHEET_URL)

        # カテゴリシートの読み込み（もしあれば）
        try:
            primary_cat_sheet = spreadsheet.worksheet("CategoryPrimary")
            primary_categories = [cat for cat in primary_cat_sheet.col_values(2) if cat]
            print(f"シート 'CategoryPrimary' から {len(primary_categories)} 件のカテゴリを読み込みました。")
        except gspread.WorksheetNotFound:
            print("警告: シート 'CategoryPrimary' が見つかりません。")

        try:
            tertiary_cat_sheet = spreadsheet.worksheet("CategoryTertiary")
            tertiary_categories = [cat for cat in tertiary_cat_sheet.col_values(2) if cat]
            print(f"シート 'CategoryTertiary' から {len(tertiary_categories)} 件のカテゴリを読み込みました。")
        except gspread.WorksheetNotFound:
            print("警告: シート 'CategoryTertiary' が見つかりません。")

        # 教師データシートの読み込み
        if ENABLE_FEW_SHOT_LEARNING:
            try:
                labeled_sheet = spreadsheet.worksheet(LABELED_SHEET_NAME)
                labeled_df = pd.DataFrame(labeled_sheet.get_all_records())
                labeled_df.dropna(how='all', inplace=True)
                # 新しいスキーマに合わせた必須列
                required_cols = ["RequirementEN", "RequirementJA", "ESGCategory", "RequirementSubcategory", "RequirementType"]
                if all(col in labeled_df.columns for col in required_cols):
                    labeled_df = labeled_df[required_cols]
                    print(f"シート '{LABELED_SHEET_NAME}' から {len(labeled_df)} 件の教師データを読み込みました。")

                    print("少数ショット学習用の「お手本」データを生成しています...")
                    sample_size = len(labeled_df) if FEW_SHOT_LIMIT <= 0 else min(len(labeled_df), FEW_SHOT_LIMIT)
                    sample_df = labeled_df.head(sample_size)

                    examples_json = []
                    for _, example_row in sample_df.iterrows():
                        draft_note_text = example_row.get('RequirementJA') or example_row.get('RequirementEN')
                        example_prompt = f"""
                        ---
                        ### お手本

                        【下書きメモの抜粋】
                        - {draft_note_text} [（教師データからの例）]

                        【出力JSON】
                        ```json
                        [
                          {{
                            "RequirementEN": "{example_row['RequirementEN']}",
                            "RequirementJA": "{example_row['RequirementJA']}",
                            "ESGCategory": "{example_row['ESGCategory']}",
                            "RequirementSubcategory": "{example_row['RequirementSubcategory']}",
                            "RequirementType": "{example_row['RequirementType']}",
                            "SourceURL": "（教師データからの例）"
                          }}
                        ]
                        ```
                        """
                        examples_json.append(example_prompt.strip())
                    few_shot_json_examples = "\n".join(examples_json)
                    print("「お手本」データの生成が完了しました。")

                else:
                    print(f"警告: '{LABELED_SHEET_NAME}'シートに必要な列 {required_cols} が不足しています。")
                    labeled_df = None
            except gspread.WorksheetNotFound:
                print(f"警告: 教師データシート '{LABELED_SHEET_NAME}' が見つかりません。")
            except Exception as e:
                print(f"教師データシート '{LABELED_SHEET_NAME}' の読み込み中にエラーが発生しました: {e}")
        else:
            print("少数ショット学習は無効に設定されています。")

        # 企業情報シートの読み込み
        worksheet = spreadsheet.worksheet(READ_SHEET_NAME)
        df = pd.DataFrame(worksheet.get_all_records())
        df = df.dropna(how='all')

        header_row = worksheet.row_values(1)
        company_col_name = header_row[ord(COMPANY_NAME_COLUMN.upper()) - 65]
        address_col_name = header_row[ord(ADDRESS_COLUMN.upper()) - 65]
        country_col_name = header_row[ord(COUNTRY_COLUMN.upper()) - 65]
        company_id_col_name = header_row[0]

        if all([company_col_name, address_col_name, country_col_name, company_id_col_name]):
             print(f"スプレッドシート '{READ_SHEET_NAME}' からデータを正常に読み込みました。")
             print(f"対象企業数: {len(df)} 件")
             print(f"ID列: '{company_id_col_name}', 企業名列: '{company_col_name}', 住所列: '{address_col_name}', 国名列: '{country_col_name}'")
             print("\n読み込んだデータ（先頭5行）:")
             print(df[[company_id_col_name, company_col_name, address_col_name, country_col_name]].head())
        else:
            print("エラー: 指定された列名がシートのヘッダーに見つかりません。")

    else:
        print("スプレッドシートのURLが入力されていないか、認証が完了していません。")

except Exception as e:
    print(f"データの読み込み中に予期せぬエラーが発生しました: {e}")

# ==============================================================================
# Helper Functions
# ==============================================================================

url_resolution_cache = {}

def generate_content_with_retry(client, model_name, prompt, tools=None, max_retries=3,
                                cached_content=None, safety_settings=None, system_instruction=None):
    """API呼び出しを自動でリトライするラッパー関数"""
    retries = 0

    while retries < max_retries:
        try:
            config_params = {}
            using_tools = False

            if cached_content:
                config_params['cached_content'] = cached_content.name
            else:
                if tools:
                    config_params['tools'] = tools
                    using_tools = True
                if system_instruction:
                    config_params['system_instruction'] = system_instruction

            config_params.update({
                "temperature": 0.0,
                "top_p": 0.0,
                "top_k": 1,
                "candidate_count": 1,
                "max_output_tokens": 32768,
            })

            if isinstance(safety_settings, list):
                config_params["safety_settings"] = safety_settings

            if not using_tools and not cached_content:
                config_params["response_mime_type"] = "application/json"
                config_params["response_schema"] = types.Schema(
                    type="ARRAY",
                    items=types.Schema(
                        type="OBJECT",
                        properties={
                            "RequirementEN": types.Schema(type="STRING"),
                            "RequirementJA": types.Schema(type="STRING"),
                            "ESGCategory": types.Schema(type="STRING"),
                            "RequirementSubcategory": types.Schema(type="STRING"),
                            "RequirementType": types.Schema(type="STRING"),
                            "SourceURL": types.Schema(type="STRING"),
                        },
                        required=["RequirementEN", "RequirementJA", "ESGCategory", "RequirementSubcategory", "RequirementType", "SourceURL"],
                    ),
                )

            print(f"  -> 送信するプロンプト(先頭500文字):\n---\n{str(prompt)[:500]}...\n---")
            start_time = time.time()
            print(f"  -> APIからの応答を待っています... (試行 {retries + 1}/{max_retries})", flush=True)

            response = client.models.generate_content(
                model=f'models/{model_name}',
                contents=prompt,
                config=config_params,
            )

            end_time = time.time()
            print(f"  -> APIから応答を受信しました。(所要時間: {end_time - start_time:.2f}秒)", flush=True)

            if hasattr(response, 'usage_metadata'):
                usage = response.usage_metadata
                total_tokens   = int(getattr(usage, 'total_token_count', 0) or 0)
                cached_tokens  = int(getattr(usage, 'cached_content_token_count', 0) or 0)
                prompt_tokens  = int(getattr(usage, 'prompt_token_count', 0) or 0)
                candidates_tok = int(getattr(usage, 'candidates_token_count', 0) or 0)
                print(f"  -> 消費トークン数: {total_tokens}", flush=True)
                if cached_tokens > 0:
                    print(f"     (内訳: キャッシュ={cached_tokens}, プロンプト={prompt_tokens}, 生成={candidates_tok})", flush=True)
                    print(f"     ✨ コンテキストキャッシュにより {cached_tokens} トークン節約されました！", flush=True)

            return response

        except Exception as e:
            retries += 1
            print(f"  -> API呼び出し中にエラーが発生しました: {e}", flush=True)
            if retries < max_retries:
                wait_time = 2 ** retries
                print(f"  -> {wait_time}秒待機して再試行します...", flush=True)
                time.sleep(wait_time)
            else:
                print("  -> 最大再試行回数に達しました。この処理をスキップします。", flush=True)
                raise e


def find_and_classify_esg_requirements(company_row, client, model_name, system_prompt_cache=None, system_prompt_text="", safety_settings=None):
    """
    2段階実行：
      Stage 1（検索あり・スキーマ無し）: Web検索で企業のESG要件・要望の「下書き」を自由テキストで収集
      Stage 2（検索なし・スキーマ有り）: 下書きを厳密JSONに整形・分類
    """
    t0 = time.time()
    print("\n企業のESG要件・要望の検索（Stage 1）→ 整形（Stage 2）を開始します...")

    company_id = company_row[company_id_col_name]
    company_name = company_row[company_col_name]
    country = company_row[country_col_name]
    address = company_row[address_col_name]

    discovered = []
    if ENABLE_PRE_DISCOVERY:
        queries = build_search_queries(company_name, address, country)[:PRE_DISCOVERY_MAX_QUERIES]
        print(f"  -> 先行ディスカバリ開始（{len(queries)}クエリ x 各{DISCOVERY_RESULT_LIMIT}件）")
        discovered = discover_sources_with_queries(
            client=client,
            model_name=model_name,
            queries=queries,
            limit=DISCOVERY_RESULT_LIMIT,
            safety_settings=safety_settings
        )
        if discovered:
            print(f"  -> ディスカバリ合計 {len(discovered)}件の候補URL")
            # ログは短縮（URLのみ）
            for i, r in enumerate(discovered, 1):
                print(f"     {i:>2}. {r['url']}")
        else:
            print("  -> ディスカバリで有効なURLは得られませんでした。")

    # URLと検索クエリの対応マップを作成
    url_to_query_map = {}
    for item in discovered:
        # URLは正規化して、後でルックアップしやすくする
        normalized_url = _normalize_url(item.get('url', ''))
        if normalized_url and normalized_url not in url_to_query_map:
            url_to_query_map[normalized_url] = item.get('query', '')

    hint_urls = _filter_and_rank_discovered(discovered, max_hint=8)
    print(f"  -> ヒントURL選抜 {len(hint_urls)}件（中継/ノイズ除去後）")
    for i, u in enumerate(hint_urls[:15], 1):
        print(f"     {i:>2}. {u}")

    stage1_prompt = f"""
あなたは企業のサステナビリティ戦略・調達方針を分析する専門家です。
以下の対象企業に**直接関係する**情報源だけをWeb検索で調べ、企業のESGに関する「要件」「要望」「目標」を箇条書きでできるだけ多く列挙してください。

### 対象企業
- 企業名: {company_name}
- 国: {country}

### 収集対象の指示
- **「〜を目指します」「〜に取り組みます」**といった将来の目標を示す記述を探してください。
- **「サプライヤーには〜を求めます」「取引先には〜を期待します」**といった、社外に対する要件を探してください。
- **「〜という方針に基づき」「〜を遵守します」**といった、企業自身の行動規範を示す記述を探してください。

### 出力フォーマット（自由テキスト）
- 箇条書きで項目を列挙してください。
- 各項目の末尾に 半角スペース + 角括弧付きURL を必ず付けてください。（例: ... [https://example.com/page]）
- 1つの情報源から複数項目が見つかった場合は、それぞれ独立した項目として列挙してください。
- **企業の製品情報、IRの財務情報、採用情報など、ESGの要件・目標に直接関係ない情報は含めないでください。**

### ヒントURL（優先的に確認し、無関係なら無視してください）
{chr(10).join(f"- {u}" for u in hint_urls)}
""".strip()

    try:
        if system_prompt_cache:
            print("  -> [Stage 1] キャッシュされたツールとシステム指示を使って下書きを収集します。")
            resp_stage1 = generate_content_with_retry(
                client=client,
                model_name=model_name,
                prompt=stage1_prompt,
                cached_content=system_prompt_cache,
                tools=None,
                safety_settings=safety_settings,
                system_instruction=None
            )
        else:
            print("  -> [Stage 1] 検索ツールで下書きを収集します。（キャッシュなし）")
            resp_stage1 = generate_content_with_retry(
                client=client,
                model_name=model_name,
                prompt=stage1_prompt,
                tools=[types.Tool(google_search=types.GoogleSearch())],
                cached_content=None,
                safety_settings=safety_settings,
                system_instruction=system_prompt_text
            )
    except Exception as e:
        print(f"  -> [Stage 1] 失敗: {e}")
        resp_stage1 = None

    notes_text = _get_text_like(resp_stage1) if resp_stage1 else ""
    if not notes_text:
        print("  -> [Stage 1] 下書きテキストが空でした。最低限のテンプレで Stage 2 に進みます。")
        notes_text = "(no findings)"

    stage2_prompt = f"""
You are a strict JSON converter and classifier for corporate ESG requirements.
Your task is to convert the DRAFT NOTES below into a structured JSON array, based on the corporate ESG requirements.

### Instructions
1.  Carefully read the DRAFT NOTES and identify each distinct ESG requirement, goal, or policy.
2.  For each item, create a JSON object by mapping the information to the required keys.
3.  The "SourceURL" must be the URL found in brackets `[...]` in the notes.
4.  Output **only** the final JSON array. If no requirements are found, return an empty array `[]`.

### Available Categories & Types
- "ESGCategory": Choose one from "E", "S", or "G".
- "RequirementType": (e.g., "Quantitative Goal", "Qualitative Policy", "Procurement Standard")
- "RequirementSubcategory": (e.g., "GHG Emissions Scope3", "Diversity & Inclusion", "Supply Chain Labor Rights")

### Few-shot examples (for format reference only):
{few_shot_json_examples}

### DRAFT NOTES (from Stage 1)
{notes_text}
""".strip()


    print("  -> [Stage 2] ツール無し+スキーマ有りで JSON 整形・分類します。")
    try:
        resp_stage2 = generate_content_with_retry(
            client=client,
            model_name=model_name,
            prompt=stage2_prompt,
            tools=None,
            cached_content=None,
            safety_settings=safety_settings,
            system_instruction="Return only a VALID JSON array of objects as specified."
        )
    except Exception as e2:
        print(f"  -> [Stage 2] 失敗: {e2}")
        resp_stage2 = None

    raw_json_text = ""
    if resp_stage2:
        raw_json_text = (resp_stage2.text or "").strip()
        if not raw_json_text:
            raw_json_text = _get_text_like(resp_stage2)
    json_text = slice_balanced_json_array(raw_json_text)

    try:
        requirements = json.loads(json_text)
    except Exception as e:
        print("  -> [Stage 2] JSONパース失敗。フォーマット修正を試みます:", e)
        fixed = (
            json_text.replace('\u201c','"').replace('\u201d','"')
                     .replace('\u2018',"'").replace('\u2019',"'")
                     .replace('\u00a0',' ')
        )
        fixed = re.sub(r',(\s*[\]}])', r'\1', fixed)
        try:
            requirements = json.loads(fixed)
        except Exception as e2:
            print("  -> [Stage 2] 再パースも失敗:", e2)
            requirements = []

    if isinstance(requirements, dict):
        requirements = [requirements]
    if not isinstance(requirements, list):
        print("  -> [Stage 2] 返却が配列ではありません。破棄して空配列にします。")
        requirements = []
    if len(requirements) > 100:
        requirements = requirements[:100]

    normed = []
    for r in requirements:
        if not isinstance(r, dict):
            continue

        raw_url_string = str(r.get("SourceURL") or "")
        cleaned_url = _get_first_valid_url_from_string(raw_url_string)

        # 最終URLを正規化し、マップから元のクエリを検索
        normalized_cleaned_url = _normalize_url(cleaned_url)
        origin_query = url_to_query_map.get(normalized_cleaned_url, "(Stage1 AIによる直接発見)")

        out = {
            "CompanyID": company_id,
            "CompanyName": company_name,
            "RequirementEN": str(r.get("RequirementEN") or ""),
            "RequirementJA": str(r.get("RequirementJA") or ""),
            "ESGCategory": str(r.get("ESGCategory") or ""),
            "RequirementSubcategory": str(r.get("RequirementSubcategory") or ""),
            "RequirementType": str(r.get("RequirementType") or ""),
            "SourceURL": cleaned_url,
            "SourceTitle": "",
            "SourceDescription": "",
            "SourceLanguage": "",
            "OriginQuery": origin_query # 貢献度分析用の列
        }
        normed.append(out)

    if normed:
        print("  -> SourceURL（採用予定）:")
        for u in sorted({row["SourceURL"] for row in normed if row["SourceURL"]}):
            print(f"     - {u}")
    else:
        print("  -> SourceURL（採用予定）は空です。")

    print(f"  -> {len(normed)} 件のESG要件・要望を抽出・分類しました。")
    print(f"  -> 統合処理完了 (所要時間: {time.time() - t0:.2f}秒)")
    return pd.DataFrame(normed)


def normalize_string(s):
    if not isinstance(s, str):
        return ""
    s = unicodedata.normalize("NFKC", s).lower()
    s = re.sub(r'[!"#$%&\'()*+,\-./:;<=>?@\[\]^_`{|}~]', ' ', s)
    s = re.sub(r'[^0-9a-z\s\u3040-\u30ff\u4e00-\u9faf]', ' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s

def is_similar(string1, string2, company_name_to_remove=""):
    a = normalize_string(string1)
    b = normalize_string(string2)

    if company_name_to_remove:
        name_norm = normalize_string(company_name_to_remove)
        if name_norm:
            a = a.replace(name_norm, "").strip()
            b = b.replace(name_norm, "").strip()

    return max(
        fuzz.token_set_ratio(a, b),
        fuzz.token_sort_ratio(a, b),
        fuzz.partial_ratio(a, b)
    )


def is_vertex_redirect(url: str) -> bool:
    return isinstance(url, str) and any(url.startswith(h) for h in VERTEX_HOSTS)

def unwrap_vertex_redirect(url: str, timeout: int = 8) -> str:
    """
    Vertex AI Search の中継URLを可能なら“生URL”に解決。
    失敗・例外時は元URLをそのまま返す（= 中継URLを残す）。
    結果はキャッシュされ、パフォーマンスを向上させます。
    エラー発生時には詳細なログを出力します。
    """
    if url in url_resolution_cache:
        return url_resolution_cache[url]

    resolved_url = url
    try:
        if not is_vertex_redirect(url):
            return url

        headers = {"User-Agent": "Mozilla/5.0", "Accept-Language": "ja,en;q=0.8"}
        r = requests.get(url, allow_redirects=True, timeout=timeout, headers=headers)
        final = r.url or url
        if not is_vertex_redirect(final):
            resolved_url = final
            return resolved_url

        text = r.text or ""

        m = re.search(r'http-equiv=["\']refresh["\'][^>]*content=["\'][^;]+;\s*url=(?P<url>https?://[^"\'<> ]+)', text, flags=re.I)
        if m:
            resolved_url = html.unescape(m.group("url"))
            return resolved_url

        m = re.search(r'window\.location(?:\.href)?\s*=\s*["\'](https?://[^"\']+)["\']', text, flags=re.I)
        if m:
            resolved_url = html.unescape(m.group(1))
            return resolved_url

        m = re.search(r'<a[^>]+href=["\'](https?://[^"\']+)["\']', text, flags=re.I)
        if m:
            resolved_url = html.unescape(m.group(1))
            return resolved_url

        print(f"  -> 警告: URL '{url}' のリダイレクトを解決できませんでした。元のURLを返します。")
        return url

    except Exception as e:
        print(f"  -> 警告: URL '{url}' の解決中にエラーが発生しました: {e}")
        resolved_url = url
        return resolved_url
    finally:
        url_resolution_cache[url] = resolved_url

def _get_first_valid_url_from_string(url_string: str) -> str:
    """
    AIが生成したカンマ区切りのURL文字列から、最初の有効な公開URLを抽出し、必要であれば解決する。
    """
    if not isinstance(url_string, str):
        return ""

    potential_urls = [u.strip() for u in re.split(r'[\s,]+', url_string) if u.strip()]

    if len(potential_urls) > 1:
        print(f"  -> 警告: 複数のURLが結合されていました。'{url_string[:100]}...'")

    for url in potential_urls:
        resolved_url = unwrap_vertex_redirect(url)

        if resolved_url.startswith(("http://", "https://")) and not is_vertex_redirect(resolved_url):
            if len(potential_urls) > 1:
                print(f"  ->      採用URL: '{resolved_url}'")
            return resolved_url

    if len(potential_urls) > 1:
        print("  ->      有効なURLが見つからなかったため、空にします。")

    return ""


# --- ヒントURL用のフィルタ＆スコアリング（中継URLを除外） --------------------
def _normalize_url(u: str) -> str:
    """末尾スラ・トラッキング除去など軽い正規化"""
    try:
        if not isinstance(u, str) or not u.startswith(("http://", "https://")):
            return ""
        p = urlparse(u)
        q = [(k, v) for (k, v) in parse_qsl(p.query, keep_blank_values=True)
             if not (k.lower().startswith("utm_") or k.lower() in {"gclid", "fbclid", "mc_cid", "mc_eid"})]
        new = p._replace(query=urlencode(q), fragment="")
        out = urlunparse(new)
        if out.endswith("//"):
            out = out[:-1]
        return out
    except Exception:
        return u or ""

def _is_public_http(u: str) -> bool:
    """中継/翻訳/旅行サイト等を除外して“本文に直接行ける”URLだけ True"""
    if not isinstance(u, str) or not u.startswith(("http://","https://")):
        return False
    host = urlparse(u).netloc.lower()

    deny_contains = (
        "vertexaisearch.", "translate.google.", "r.jina.ai",
        "webcache.googleusercontent.com", "duckduckgo.com", "bing.com",
        "search.yahoo.", "sitelinks.", "lmgtfy",
        "booking.com", "airbnb.", "workin.space", "coworkinghub.",
    )
    if any(d in host for d in deny_contains):
        return False
    return True

def _score_domain(u: str) -> int:
    """優先度付け（高いほど上位）。一次・公式・公的・認証等を厚めに。"""
    host = urlparse(u).netloc.lower()
    path = urlparse(u).path.lower()

    # NOTE: ドメインのスコアリングは、企業調査用に調整が必要な場合があります
    if any(s in host for s in (
        ".gov", ".org", ".edu"
    )) or path.endswith(".pdf"):
        return 90
    if any(s in host for s in ("csr-reports.com", "sustainability-reports.com", "globalreporting.org")):
        return 80
    return 50

def _filter_and_rank_discovered(items, max_hint=15):
    """
    discovered: [{"title": ..., "url": ..., "snippet": ...}, ...]
    を受け取り、ヒント用URL（中継除外＆正規化＆スコア順）に整形
    """
    seen = set()
    cleaned = []
    for r in items or []:
        u = (r or {}).get("url", "")
        if not _is_public_http(u):
            continue
        u = _normalize_url(u)
        if not u or u in seen:
            continue
        seen.add(u)
        cleaned.append((u, _score_domain(u)))

    cleaned.sort(key=lambda t: (-t[1], len(t[0])))
    return [u for (u, _) in cleaned[:max_hint]]
# ---------------------------------------------------------------------------



def slice_balanced_json_array(text: str) -> str:
    """最初の '[' から括弧・クォートのバランスが取れる最後の位置までを安全に切り出す"""
    if not isinstance(text, str):
        return "[]"
    t = text.strip()
    t = t.replace("```json", "```")
    t = t.strip("` \n\r\t")

    start = t.find('[')
    if start == -1:
        return "[]"

    depth = 0
    in_str = False
    esc = False
    quote = ''
    last_ok = -1

    for i in range(start, len(t)):
        ch = t[i]
        if in_str:
            if esc:
                esc = False
            elif ch == '\\':
                esc = True
            elif ch == quote:
                in_str = False
            continue
        else:
            if ch in ('"', "'"):
                in_str = True
                quote = ch
            elif ch in '[{':
                depth += 1
            elif ch in ']}':
                depth -= 1
                if depth == 0:
                    last_ok = i
                    break

    if last_ok != -1:
        return t[start:last_ok+1]
    last_brace = t.rfind('}')
    if last_brace > start:
        return t[start:last_brace+1] + ']'
    return "[]"

# 企業のESG要件・要望を検索するためのキーワード戦略
SEARCH_STRATEGIES = {
    "default": { # English as default
        "base": [
            "sustainable procurement", "supplier code of conduct", "responsible sourcing"
        ],
        "specific_esg": [
            "ESG goals", "sustainability targets", "human rights policy", "environmental policy", "TCFD report"
        ]
    },
    "ja": {
        "base": [
            "ESG調達", "サステナビリティ調達", "サプライヤー行動規範", "CSR調達"
        ],
        "specific_esg": [
            "ESG目標", "サステナビリティ目標", "人権方針", "環境方針", "TCFD"
        ]
    }
}

def build_search_queries(company_name: str, address: str, country: str) -> list:
    """
    企業名、国名に基づき、最適な検索クエリのリストを動的に生成する。
    """
    name_q = f'"{company_name}"' if company_name else ""
    if not name_q: return []

    strategy_key = "ja" if "japan" in (country or "").lower() or "日本" in (country or "") else "default"
    strategy = SEARCH_STRATEGIES.get(strategy_key, SEARCH_STRATEGIES["default"])

    keywords = []
    keywords.extend(strategy.get("base", []))
    keywords.extend(strategy.get("specific_esg", []))

    queries = [f"{name_q} {keyword}" for keyword in keywords]

    # 必要に応じて、国別の規制や報告義務に関するキーワードを追加 (例: Modern Slavery Act)
    c = (country or "").strip().lower()
    if any(k in c for k in ["uk", "united kingdom", "australia"]):
        queries.append(f"{name_q} Modern Slavery Act")

    # 重複・空要素を除去して返す
    out = [q.strip() for q in queries if q.strip()]
    out = list(dict.fromkeys(out))  # 順序維持で重複排除
    return out


def _safe_unwrap(u: str) -> str:
    try:
        return unwrap_vertex_redirect(u) if is_vertex_redirect(u) else u
    except Exception:
        return u

def discover_sources_with_queries(client, model_name, queries: list, limit: int, safety_settings=None):
    """
    各クエリで google_search ツールを使い、上位URLを収集して返す。
    ログも詳細に出す。
    戻り値に元のクエリを含める
    """
    tool = types.Tool(google_search=types.GoogleSearch())
    found = []
    for idx, q in enumerate(queries, 1):
        prompt = f"""
Use your web search tool to search **exactly** this query and return top {limit} pages as a JSON array.
Return only JSON. Each item: {{"title": "...", "url": "...", "snippet": "..."}}
Query: {q}
"""
        try:
            resp = generate_content_with_retry(
                client=client,
                model_name=model_name,
                prompt=prompt,
                tools=[tool],
                cached_content=None,
                safety_settings=safety_settings,
                system_instruction=None
            )
            txt = (resp.text or "").strip().replace("```json", "```").strip("` \n\r\t")
            arr_text = slice_balanced_json_array(txt)
            results = json.loads(arr_text)
            if isinstance(results, dict):
                results = [results]
            if isinstance(results, list):
                count = 0
                for r in results[:limit]:
                    url = (r or {}).get("url", "")
                    if url and url.startswith(("http://", "https://")):
                        found.append({
                            "title": (r or {}).get("title", ""),
                            "url": _safe_unwrap(url),
                            "snippet": (r or {}).get("snippet", ""),
                            "query": q # どのクエリ由来かを記録
                        })
                        count += 1
            # ログ
            print(f"  -> [Discovery {idx}/{len(queries)}] \"{q}\" → {len(results) if isinstance(results, list) else 0}件（採用{count}）")
            # ログは短縮
            for r in (results or [])[:limit]:
                u = (r or {}).get("url", "")
                if u:
                    print(f"       - {u}")

        except Exception as e:
            print(f"  -> [Discovery {idx}/{len(queries)}]  失敗: {e}")
    # 重複排除（URL単位）
    uniq = []
    seen = set()
    for r in found:
        if r["url"] not in seen:
            uniq.append(r)
            seen.add(r["url"])
    return uniq

def _get_text_like(resp) -> str:
    """
    response.text が空でも candidates/parts から最終テキストを救出。
    JSON限定ではなく “自由テキストの下書き” 用。
    """
    try:
        s = (resp.text or "").strip()
        if s:
            return s
        cand = resp.candidates[0]
        parts = getattr(cand.content, "parts", []) or []
        for p in reversed(parts):
            t = getattr(p, "text", None)
            if isinstance(t, str) and t.strip():
                return t.strip()
        for p in parts:
            inline = getattr(p, "inline_data", None)
            if inline and getattr(inline, "mime_type", "") in ("text/plain", "application/json"):
                try:
                    return inline.data.decode("utf-8").strip()
                except Exception:
                    pass
    except Exception:
        pass
    return ""

# ==============================================================================
# Main Execution Flow
# ==============================================================================
if df is not None and not df.empty:
    system_prompt_cache = None
    system_prompt_for_caching = ""
    try:
        safety_settings = [
            types.SafetySetting(category="HARM_CATEGORY_HARASSMENT",        threshold="BLOCK_NONE"),
            types.SafetySetting(category="HARM_CATEGORY_HATE_SPEECH",       threshold="BLOCK_NONE"),
            types.SafetySetting(category="HARM_CATEGORY_SEXUALLY_EXPLICIT", threshold="BLOCK_NONE"),
            types.SafetySetting(category="HARM_CATEGORY_DANGEROUS_CONTENT", threshold="BLOCK_NONE"),
        ]


        print(f"全ての分析にユーザー選択モデル '{MODEL_NAME}' を使用します。", flush=True)

        search_tool = types.Tool(google_search=types.GoogleSearch())

        # このシステムプロンプトは現在直接は使われていないが、将来的な拡張やキャッシュのために残す
        system_prompt_for_caching = f"""
        あなたは企業のサステナビリティ戦略・調達方針を分析する専門家です。
        あなたのタスクは、指定された企業に関するESGの「要件」「要望」「目標」をWebから抽出し、分類することです。

        ### タスクの実行手順
        1.  **Web検索**: Web検索ツールを駆使して、対象企業に関する信頼できる情報源（公式サイト、サステナビリティレポート、CSR方針ページなど）を見つけ出してください。
        2.  **要件の抽出**: 見つけた情報源から、企業が掲げる将来の目標、サプライヤーやパートナーに対する要件、または遵守を掲げる方針を具体的に抽出してください。「〜を目指します」「サプライヤーには〜を求めます」といった表現に着目してください。
        3.  **分類**: 抽出した各要件について、適切なカテゴリ分類を行ってください。
        4.  **出典の明記**: 各要件がどのWebページの情報に基づいているか、そのURLを必ず明記してください。

        ### 回答形式
        最終的な回答は、指示されたJSONスキーマに厳密に従ったJSON配列のみを出力してください。JSON以外のテキストは含めないでください。
        """

        if CACHE_TTL_HOURS > 0 and client:
            print(f"\n分析用のコンテキストキャッシュを作成します（有効時間: {CACHE_TTL_HOURS}時間）...", flush=True)

            try:
                ttl_seconds = int(pd.to_timedelta(CACHE_TTL_HOURS, 'h').total_seconds())

                cache_config = types.CreateCachedContentConfig(
                    system_instruction=system_prompt_for_caching,
                    tools=[types.Tool(google_search=types.GoogleSearch())],
                    ttl=f"{ttl_seconds}s"
                )

                system_prompt_cache = client.caches.create(
                    model=f'models/{MODEL_NAME}',
                    config=cache_config
                )
                print(f"コンテキストキャッシュの作成に成功しました。キャッシュ名: {system_prompt_cache.name}", flush=True)
            except Exception as e:
                print(f"コンテキストキャッシュの作成中にエラーが発生しました: {e}", flush=True)
                system_prompt_cache = None
        else:
            print("\nコンテキストキャッシュは無効に設定されています。", flush=True)


        print("Geminiモデルの初期化が完了しました。", flush=True)

        print(f"\n書き込み先シート '{WRITE_SHEET_NAME}' を準備します...", flush=True)
        try:
            write_worksheet = spreadsheet.worksheet(WRITE_SHEET_NAME)
            print("  -> 既存のデータに追記するモードで実行します。", flush=True)

            final_columns = [
                "CompanyID", "CompanyName", "RequirementEN", "RequirementJA",
                "ESGCategory", "RequirementSubcategory", "RequirementType", "SourceURL",
                "SourceTitle", "SourceDescription", "SourceLanguage"
            ]
            header = write_worksheet.row_values(1) if write_worksheet.row_count > 0 else []
            if header != final_columns:
                print("  -> ヘッダー行を書き込みます...", flush=True)
                write_worksheet.update('A1', [final_columns])

        except gspread.WorksheetNotFound:
            write_worksheet = spreadsheet.add_worksheet(title=WRITE_SHEET_NAME, rows="1000", cols="20")
            final_columns = [
                "CompanyID", "CompanyName", "RequirementEN", "RequirementJA",
                "ESGCategory", "RequirementSubcategory", "RequirementType", "SourceURL",
                "SourceTitle", "SourceDescription", "SourceLanguage"
            ]
            write_worksheet.append_row(final_columns)
            print(f"  -> シート '{WRITE_SHEET_NAME}' を新しく作成し、ヘッダーを書き込みました。", flush=True)
        except Exception as e:
            print(f"  -> 書き込み先シートの準備中にエラーが発生しました: {e}")


        print("書き込み先シートの準備が完了しました。", flush=True)

    except Exception as e:
        print(f"Geminiモデルまたはスプレッドシートの初期化中にエラーが発生しました: {e}", flush=True)
        client = None

    if client:
        for index, company_row in df.iterrows():
            company_name = company_row[company_col_name]
            print(f"\n{'='*20} 企業 {index + 1}/{len(df)}: '{company_name}' の処理を開始 {'='*20}", flush=True)

            requirements_df = find_and_classify_esg_requirements(
                company_row, client, MODEL_NAME, system_prompt_cache, system_prompt_for_caching, safety_settings
            )

            if not requirements_df.empty:
                print("\n  -> 重複チェックを実行します...", flush=True)
                start_time_dedup = time.time()

                requirements_df = requirements_df.drop_duplicates(
                    subset=["RequirementEN", "RequirementJA", "ESGCategory", "RequirementSubcategory", "RequirementType", "SourceURL"],
                    keep="first"
                )

                cleaned_rows = []
                for url, sub in requirements_df.groupby("SourceURL", dropna=False):
                    kept = []
                    for _, r in sub.iterrows():
                        name_en = r.get('RequirementEN', '') or ''
                        name_ja = r.get('RequirementJA', '') or ''
                        cat1 = (r.get('ESGCategory','') or '').strip()
                        cat3 = (r.get('RequirementSubcategory','') or '').strip()

                        dup = False
                        for k in kept:
                            if (cat1, cat3) != (k.get('ESGCategory','').strip(), k.get('RequirementSubcategory','').strip()):
                                continue

                            score = max(
                                is_similar(name_en, k.get('RequirementEN',''), company_name_to_remove=company_name),
                                is_similar(name_ja, k.get('RequirementJA',''), company_name_to_remove=company_name)
                            )
                            if score >= SIMILARITY_THRESHOLD:
                                dup = True
                                break

                        if not dup:
                            kept.append(r)

                    cleaned_rows.extend(kept)

                requirements_df = pd.DataFrame(cleaned_rows).reset_index(drop=True)


                end_time_dedup = time.time()
                print(f"  -> 重複チェック完了 (所要時間: {end_time_dedup - start_time_dedup:.2f}秒)")

                # キーワード別貢献度分析ログを出力
                if not requirements_df.empty and 'OriginQuery' in requirements_df.columns:
                    print("\n  -> キーワード別ESG要件・要望貢献度:")
                    temp_df = requirements_df.copy()
                    temp_df['OriginQuery'] = temp_df['OriginQuery'].replace({"": "(不明)", "(Stage1 AIによる直接発見)": "(Stage1 AIによる直接発見)"})

                    contribution = temp_df['OriginQuery'].value_counts().reset_index()
                    contribution.columns = ['キーワード', '要件数']
                    contribution = contribution.sort_values(by='要件数', ascending=False)

                    for _, row in contribution.iterrows():
                        print(f"     - {row['要件数']:>3}件: {row['キーワード']}")

                if not requirements_df.empty:
                    # スプレッドシートに書き込む前に不要な列を削除
                    if 'OriginQuery' in requirements_df.columns:
                      requirements_df = requirements_df.drop(columns=['OriginQuery'])

                    for col in final_columns:
                        if col not in requirements_df.columns:
                            requirements_df[col] = ""
                    requirements_df = requirements_df[final_columns]

                    requirements_df = requirements_df.fillna('')

                    def _cleanup_url(u: str) -> str:
                        u = (u or "").strip()
                        return unwrap_vertex_redirect(u) if is_vertex_redirect(u) else u
                    requirements_df["SourceURL"] = requirements_df["SourceURL"].map(_cleanup_url)

                    rows_to_append = requirements_df.values.tolist()

                    print("\n  -> シートの入力規則を保持するため、セル範囲を指定して追記します...", flush=True)
                    start_time_write = time.time()

                    last_row = len(write_worksheet.col_values(1))
                    start_row = last_row + 1
                    end_row = last_row + len(rows_to_append)
                    update_range = f'A{start_row}:{gspread.utils.rowcol_to_a1(end_row, len(final_columns))}'

                    write_worksheet.update(
                        range_name=update_range,
                        values=rows_to_append,
                        value_input_option='USER_ENTERED'
                    )

                    end_time_write = time.time()
                    print(f"  -> '{company_name}' の結果 {len(rows_to_append)} 件をシートに書き込みました。(所要時間: {end_time_write - start_time_write:.2f}秒)")

            else:
                print(f"\n  -> '{company_name}' に関連する具体的なESG要件・要望は見つかりませんでした。", flush=True)

        print(f"\n{'='*20} 全ての企業の処理が完了しました。 {'='*20}")

    else:
        print("\nモデルが初期化されていないため、メイン処理をスキップしました。", flush=True)


else:
    print("\n処理対象の企業情報が読み込めなかったため、処理を終了します。", flush=True)



     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 241.8/241.8 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 159.9/159.9 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 26.2 MB/s eta 0:00:00

ライブラリのインストール・更新が完了しました。
google-genai version: 1.33.0
認証中にエラーが発生しました。ランタイム再起動後の2回目の実行であることを確認してください。エラー: Error: credential propagation was unsuccessful
Gemini APIキーの読み込みと設定が完了しました。
スプレッドシートのURLが入力されていないか、認証が完了していません。

処理対象の企業情報が読み込めなかったため、処理を終了します。
